# YOLOv8 Object Detection Training

This notebook trains a YOLOv8 model for object detection on Google Colab.

## 1. Install Required Libraries

In [26]:
!pip install -q ultralytics opencv-python pillow matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.8 MB/s eta 0:00:00a 0:00:01


## 2. Mount Google Drive

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2.5. Extract Dataset from Zip File

In [34]:
import zipfile
import os
import glob

# Find and extract the dataset zip file
gdrive_data_path = '/content/drive/MyDrive/Colab_Notebooks/data'
zip_files = glob.glob(os.path.join(gdrive_data_path, '*.zip'))

if zip_files:
    zip_path = zip_files[0]
    print(f"Found zip file: {zip_path}")
    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(gdrive_data_path)

    print("✓ Dataset extracted successfully")

    # List extracted contents
    extracted_items = os.listdir(gdrive_data_path)
    print(f"\nExtracted items: {extracted_items}")
else:
    print(f"❌ No zip files found in {gdrive_data_path}")
    print("Make sure CVH-ppe-prueba1.v1i.yolov8.zip is in the data folder")

Found zip file: /content/drive/MyDrive/Colab_Notebooks/data/data_horse_human.zip
Extracting dataset...
✓ Dataset extracted successfully

Extracted items: ['valid', 'test', 'train', 'README.dataset.txt', 'README.roboflow.txt', 'data_horse_human.zip', 'rock__sisor_paper.zip', 'construction safety.v2i.yolov8.zip', 'data.yaml', 'data_유전자_데이터.zip', 'CatDog_Segmentation.zip', 'lung_segmentation.zip', 'horse', 'human']


## 2.7. Verify Dataset Structure

In [37]:
import os
import glob

gdrive_data_path = '/content/drive/MyDrive/Colab_Notebooks/data'

# Recursively find data.yaml and train/images directory
print(f"Searching for data.yaml in {gdrive_data_path}...\n")

data_yaml_locations = glob.glob(os.path.join(gdrive_data_path, '**/data.yaml'), recursive=True)

print(f"Found {len(data_yaml_locations)} data.yaml file(s):")
for yaml_path in data_yaml_locations:
    print(f"  ✓ {yaml_path}")

if data_yaml_locations:
    # Use the first found data.yaml
    actual_data_yaml = data_yaml_locations[0]
    actual_data_dir = os.path.dirname(actual_data_yaml)

    print(f"\n✓ Using: {actual_data_yaml}")
    print(f"Dataset root: {actual_data_dir}")

    # List contents of the dataset directory
    print(f"\nContents of {actual_data_dir}:")
    for item in os.listdir(actual_data_dir):
        item_path = os.path.join(actual_data_dir, item)
        if os.path.isdir(item_path):
            # Count files in subdirectory
            try:
                file_count = len(os.listdir(item_path))
                print(f"  📁 {item}/ ({file_count} items)")
            except:
                print(f"  📁 {item}/")
        else:
            print(f"  📄 {item}")
else:
    print("❌ No data.yaml found!")

Searching for data.yaml in /content/drive/MyDrive/Colab_Notebooks/data...

Found 1 data.yaml file(s):
  ✓ /content/drive/MyDrive/Colab_Notebooks/data/data.yaml

✓ Using: /content/drive/MyDrive/Colab_Notebooks/data/data.yaml
Dataset root: /content/drive/MyDrive/Colab_Notebooks/data

Contents of /content/drive/MyDrive/Colab_Notebooks/data:
  📁 valid/ (2 items)
  📁 test/ (2 items)
  📁 train/ (2 items)
  📄 README.dataset.txt
  📄 README.roboflow.txt
  📄 data_horse_human.zip
  📄 rock__sisor_paper.zip
  📄 construction safety.v2i.yolov8.zip
  📄 data.yaml
  📄 data_유전자_데이터.zip
  📄 CatDog_Segmentation.zip
  📄 lung_segmentation.zip
  📁 horse/ (500 items)
  📁 human/ (527 items)


## 2.9. Find Correct Data Paths

In [38]:
import os
import glob

gdrive_data_path = '/content/drive/MyDrive/Colab_Notebooks/data'

# Find all train/images directories recursively
print("🔍 Searching for train/images directory...\n")
train_images_paths = glob.glob(os.path.join(gdrive_data_path, '**/train/images'), recursive=True)

if train_images_paths:
    print(f"✓ Found {len(train_images_paths)} train/images folder(s):")
    for path in train_images_paths:
        img_count = len(glob.glob(os.path.join(path, '*')))
        print(f"  ✓ {path}")
        print(f"    → Images: {img_count} files\n")

    # Get the parent directory of train/images
    train_images_path = train_images_paths[0]
    dataset_root = os.path.dirname(os.path.dirname(train_images_path))

    print(f"✓ Dataset root will be: {dataset_root}")
else:
    print("❌ No train/images folder found!")
    print(f"📁 Contents of {gdrive_data_path}:")

    def show_tree(path, prefix="", max_depth=3, current_depth=0):
        if current_depth >= max_depth:
            return
        try:
            items = sorted(os.listdir(path))
            for i, item in enumerate(items):
                item_path = os.path.join(path, item)
                is_last = i == len(items) - 1
                print(f"{prefix}{'└── ' if is_last else '├── '}{item}")
                if os.path.isdir(item_path) and not item.startswith('.'):
                    extension = "    " if is_last else "│   "
                    show_tree(item_path, prefix + extension, max_depth, current_depth + 1)
        except:
            pass

    show_tree(gdrive_data_path)

🔍 Searching for train/images directory...

✓ Found 1 train/images folder(s):
  ✓ /content/drive/MyDrive/Colab_Notebooks/data/train/images
    → Images: 0 files

✓ Dataset root will be: /content/drive/MyDrive/Colab_Notebooks/data


## 3. DETAILED Folder Structure Analysis

In [41]:
import os
import subprocess

gdrive_data_path = '/content/drive/MyDrive/Colab_Notebooks/data'

print("=" * 80)
print("📂 COMPLETE DIRECTORY TREE")
print("=" * 80)

# Use tree command if available, otherwise use custom function
try:
    result = subprocess.run(['tree', '-L', '3', gdrive_data_path],
                          capture_output=True, text=True, timeout=5)
    print(result.stdout)
except:
    # Fallback: custom tree printer
    def print_tree(path, prefix="", max_depth=3, current_depth=0):
        if current_depth >= max_depth:
            return
        try:
            items = sorted(os.listdir(path))
            # Filter hidden files
            items = [i for i in items if not i.startswith('.')]

            for i, item in enumerate(items):
                item_path = os.path.join(path, item)
                is_last = i == len(items) - 1
                current_prefix = "└── " if is_last else "├── "
                print(f"{prefix}{current_prefix}{item}")

                if os.path.isdir(item_path):
                    next_prefix = prefix + ("    " if is_last else "│   ")
                    print_tree(item_path, next_prefix, max_depth, current_depth + 1)
                elif os.path.isfile(item_path):
                    # Show file size
                    try:
                        size = os.path.getsize(item_path)
                        size_str = f"{size/1024:.1f}KB" if size < 1024*1024 else f"{size/(1024*1024):.1f}MB"
                        print(f"{prefix}    ({size_str})")
                    except:
                        pass
        except PermissionError:
            print(f"{prefix}    [Permission Denied]")

    print(gdrive_data_path + "/")
    print_tree(gdrive_data_path)

print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)

# Find key directories
for root, dirs, files in os.walk(gdrive_data_path):
    # Show short paths from root
    rel_path = os.path.relpath(root, gdrive_data_path)
    if rel_path == '.':
        rel_path = '[root]'

    # Look for images and labels
    img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    label_files = [f for f in files if f.lower().endswith('.txt')]
    yaml_files = [f for f in files if f.lower().endswith(('.yaml', '.yml'))]

    if img_files:
        print(f"📷 {rel_path}/: {len(img_files)} images")
    if label_files:
        print(f"📝 {rel_path}/: {len(label_files)} labels")
    if yaml_files:
        print(f"⚙️  {rel_path}/: {yaml_files}")

📂 COMPLETE DIRECTORY TREE
/content/drive/MyDrive/Colab_Notebooks/data/
├── CatDog_Segmentation.zip
    (154.4MB)
├── README.dataset.txt
    (0.1KB)
├── README.roboflow.txt
    (1.2KB)
├── construction safety.v2i.yolov8.zip
    (147.3MB)
├── data.yaml
    (0.4KB)
├── data_horse_human.zip
    (21.5MB)
├── data_유전자_데이터.zip
    (84.3MB)
├── horse
│   ├── horse01-0.jpg
│       (22.6KB)
│   ├── horse01-1.jpg
│       (19.1KB)
│   ├── horse01-2.jpg
│       (16.1KB)
│   ├── horse01-3.jpg
│       (16.4KB)
│   ├── horse01-4.jpg
│       (17.8KB)
│   ├── horse01-5.jpg
│       (19.5KB)
│   ├── horse01-6.jpg
│       (21.9KB)
│   ├── horse01-7.jpg
│       (22.9KB)
│   ├── horse01-8.jpg
│       (22.1KB)
│   ├── horse01-9.jpg
│       (23.7KB)
│   ├── horse02-0.jpg
│       (22.8KB)
│   ├── horse02-1.jpg
│       (19.5KB)
│   ├── horse02-2.jpg
│       (17.0KB)
│   ├── horse02-3.jpg
│       (17.5KB)
│   ├── horse02-4.jpg
│       (18.5KB)
│   ├── horse02-5.jpg
│       (20.0KB)
│   ├── horse02-6.jpg
│ 

## 3. Setup Environment and Train Model

In [ ]:
import os
import yaml
import glob
from ultralytics import YOLO

# Output setup
output_dir = '/content/drive/MyDrive/Colab_Notebooks/results'
os.makedirs(output_dir, exist_ok=True)

gdrive_data_path = '/content/drive/MyDrive/Colab_Notebooks/data'

print("=" * 70)
print("🔍 STEP 1: Locating Dataset Files")
print("=" * 70)

# Strategy 1: Find train/images directory
train_images_paths = glob.glob(os.path.join(gdrive_data_path, '**/train/images'), recursive=True)

if train_images_paths:
    print(f"✓ Found train/images at: {train_images_paths[0]}")
    train_images_path = train_images_paths[0]
    # Get dataset root (two levels up from train/images)
    dataset_root = os.path.dirname(os.path.dirname(train_images_path))
else:
    print(f"❌ train/images not found in {gdrive_data_path}")
    raise FileNotFoundError("Cannot find train/images directory in dataset")

print(f"✓ Dataset root: {dataset_root}")

# Find or create data.yaml
data_yaml_path = os.path.join(dataset_root, 'data.yaml')

if os.path.exists(data_yaml_path):
    print(f"✓ Found data.yaml at: {data_yaml_path}")
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f) or {}
else:
    print(f"⚠ data.yaml not found, creating new one...")
    data_config = {}

print("\n" + "=" * 70)
print("🔍 STEP 2: Finding and Verifying Dataset Paths")
print("=" * 70)

# Update path to dataset root
data_config['path'] = dataset_root
print(f"✓ Dataset path: {dataset_root}\n")

# TRAIN: Required
train_path = os.path.join(dataset_root, 'train/images')
data_config['train'] = train_path
if os.path.exists(train_path):
    img_count = len(glob.glob(os.path.join(train_path, '*.*')))
    print(f"✓ TRAIN:  {train_path}")
    print(f"         → {img_count} images\n")
else:
    raise FileNotFoundError(f"❌ train/images not found at {train_path}")

# VAL: Try multiple names
val_path = None
for val_name in ['val', 'valid']:
    candidate = os.path.join(dataset_root, f'{val_name}/images')
    if os.path.exists(candidate):
        val_path = candidate
        print(f"✓ VAL:    {val_path}")
        img_count = len(glob.glob(os.path.join(val_path, '*.*')))
        print(f"         → {img_count} images\n")
        break

if val_path:
    data_config['val'] = val_path
else:
    print(f"⚠ VAL:    No val/images or valid/images found")
    print(f"         → Removing val from config (will use train for validation)\n")
    # Remove val if it exists in config
    if 'val' in data_config:
        del data_config['val']

# TEST: Optional
test_image_paths = glob.glob(os.path.join(dataset_root, '**/test/images'), recursive=True)
if test_image_paths:
    test_path = test_image_paths[0]
    data_config['test'] = test_path
    print(f"✓ TEST:   {test_path}")
    img_count = len(glob.glob(os.path.join(test_path, '*.*')))
    print(f"         → {img_count} images\n")
else:
    print(f"⚠ TEST:   No test/images found (optional)\n")
    if 'test' in data_config:
        del data_config['test']

# Classes and num_classes
if 'nc' not in data_config:
    data_config['nc'] = 4
if 'names' not in data_config:
    data_config['names'] = ['0', '1', '2', 'mask']

print("=" * 70)
print("📝 Final data.yaml Configuration")
print("=" * 70)
for key, value in data_config.items():
    if key != 'names':
        print(f"{key:10s}: {value}")
    else:
        print(f"{key:10s}: {value}")

# Save updated yaml
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"\n✓ Saved: {data_yaml_path}")

print("\n" + "=" * 70)
print("🚀 STEP 3: Starting YOLOv8 Training")
print("=" * 70)

# Load YOLOv8 model
model = YOLO("yolov8n.pt")

# Train the model
results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=8,
    project=output_dir,
    name='detection_model',
    device='cpu'  # Colab will auto-use GPU if available
)

print("\n✓✓✓ Training completed! ✓✓✓")

🔍 STEP 1: Locating Dataset Files
✓ Found train/images at: /content/drive/MyDrive/Colab_Notebooks/data/train/images
✓ Dataset root: /content/drive/MyDrive/Colab_Notebooks/data
✓ Found data.yaml at: /content/drive/MyDrive/Colab_Notebooks/data/data.yaml

🔍 STEP 2: Verifying Dataset Paths
✓ train : /content/drive/MyDrive/Colab_Notebooks/data/train/images
           → 0 images found
⚠ val   : /content/drive/MyDrive/Colab_Notebooks/data/val/images
           → NOT FOUND (may cause training issues)
✓ test  : /content/drive/MyDrive/Colab_Notebooks/data/test/images
           → 340 images found
✓ labels: /content/drive/MyDrive/Colab_Notebooks/data/train/labels
         → 0 label files found

✓ Updated data.yaml saved to: /content/drive/MyDrive/Colab_Notebooks/data/data.yaml

🚀 STEP 3: Starting YOLOv8 Training
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=

RuntimeError: Dataset '/content/drive/MyDrive/Colab_Notebooks/data/data.yaml' error ❌ Dataset '/content/drive/MyDrive/Colab_Notebooks/data/data.yaml' images not found, missing path '/content/drive/MyDrive/Colab_Notebooks/data/val/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

## 4. View Training Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Display training metrics
metric_images = glob.glob(os.path.join(output_dir, 'detection_model', '*.png'))

if metric_images:
    print("Training Results:")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, img_path in enumerate(metric_images[:4]):
        img = mpimg.imread(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(os.path.basename(img_path))
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print(f"No metrics images found in {output_dir}/detection_model/")